<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/Makro_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%capture
# Install necessary packages for Selenium in Colab
!apt-get update
!apt-get install -y chromium-browser chromium-chromedriver
!pip install selenium bs4 pandas
!pip install playwright beautifulsoup4 polars
!playwright install chromium
!playwright install-deps

# 1. Clear out the broken Ubuntu packages
!apt-get remove -y chromium-browser chromium-chromedriver

# 2. Download and install the official Google Chrome stable version
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!sh -c 'echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list'
!apt-get -y update
!apt-get install -y google-chrome-stable

# 3. Install Python dependencies including the auto-installer
!pip install selenium bs4 pandas chromedriver-autoinstaller
!pip install polars xlsxwriter fastexcel
!pip install playwright beautifulsoup4 polars
!playwright install
!pip install itables
!playwright install-deps
!pip install curl_cffi

%%capture
# 1. Install all dependencies including pandas
!pip install xlsxwriter scrapling patchright msgspec browserforge nest_asyncio polars -q
!patchright install chromium
!patchright install-deps

### Import Lib

In [2]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import xlsxwriter
import datetime
from datetime import date
import IPython.display as display
import datetime
today_date = datetime.datetime.now().strftime("%Y-%m-%d")
print("Today is",today_date)

Today is 2026-04-10


In [14]:
from google.colab import data_table
data_table.enable_dataframe_formatter()

In [3]:
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import polars as pl
import asyncio
import re

async def scrape_makro_spa_clicker(start_url: str):
    extracted_data = []
    seen_names = set() # Track names so we don't scrape duplicates if the page is slow

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--disable-blink-features=AutomationControlled"]
        )
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1920, "height": 1080}
        )

        page = await context.new_page()
        print("Walking through Makro's front door...")

        # Load the base URL ONCE
        await page.goto(start_url, wait_until="networkidle")
        await asyncio.sleep(6) # Wait for initial hydration

        assume_MAX_page = 16

        for page_num in range(1, assume_MAX_page + 1):
            print(f"Scraping page {page_num}...")

            # Scroll to the bottom to force lazy-loaded images and prices to render
            await page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            await asyncio.sleep(2)

            # Take a snapshot of the DOM
            html_content = await page.content()
            soup = BeautifulSoup(html_content, "html.parser")

            # Makro wraps product cards in <a> tags linking to /p/ (product pages)
            product_cards = soup.find_all("a", href=lambda href: href and "/p/" in href)

            new_items_found = 0

            for card in product_cards:
                texts = list(card.stripped_strings)
                if len(texts) < 3:
                    continue

                # 1. Hunt for the Product Name
                name = None
                for text in texts:
                    if len(text) > 10 and "฿" not in text and "points" not in text.lower() and "Today" not in text:
                        name = text
                        break

                if not name or name in seen_names:
                    continue

                # 2. Hunt for the Prices (Hyper-Resilient Float Extractor)
                prices = []
                for text in texts:
                    if "buy" in text.lower() or "get" in text.lower() or "point" in text.lower():
                        continue

                    # Strip out currency symbols and commas
                    clean_text = text.replace("฿", "").replace(",", "").strip()

                    try:
                        val = float(clean_text)
                        # Safeguard: Ensure it's an actual price, not a quantity like "3 options"
                        if 5 < val < 10000:
                            prices.append(val)
                    except ValueError:
                        pass

                if not prices:
                    continue

                # 3. The E-Commerce Rule: Smallest is promo, Largest is original
                promo_price = min(prices)
                original_price = max(prices)

                # 4. Hunt for the Discount Condition (e.g., "2 - 2 units")
                condition = None

                # Primary strategy: Target the robust data-test-id you identified
                condition_tag = card.find(lambda tag: tag.has_attr("data-test-id") and "_lbl_buy_more" in tag["data-test-id"])

                if condition_tag:
                    condition = condition_tag.get_text(strip=True)
                else:
                    # Fallback strategy: Read through strings to catch anomalies like "3+ units"
                    for text in texts:
                        if "unit" in text.lower() and any(char.isdigit() for char in text):
                            condition = text
                            break

                # 5. Append everything to the dataset
                extracted_data.append({
                    "name": name,
                    "promotion_price": promo_price,
                    "original_price": original_price,
                    "condition": condition # Pushed the new column here
                })

                seen_names.add(name)
                new_items_found += 1

            print(f"  -> Extracted {new_items_found} new products.")

            # 6. THE SPA CLICKER
            if page_num < assume_MAX_page:
                try:
                    # Find the pagination "Next" button and click it
                    next_button = page.locator("text=Next").first

                    if await next_button.is_visible():
                        await next_button.click()
                        print("  -> Clicked 'Next'. Waiting for SPA to load...")
                        await asyncio.sleep(4) # Give React time to swap the products
                    else:
                        print("  -> 'Next' button not visible. Reached the end of the catalog!")
                        break
                except Exception as e:
                    print(f"  -> Pagination stopped: {e}")
                    break

        await browser.close()

    # --- POLARS CLEANUP ---
    df = pl.DataFrame(extracted_data)

    if not df.is_empty():
        df = df.unique(subset=["name"], maintain_order=True)

        # Nullify fake promotions
        df = df.with_columns(
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )

        # Calculate the Discount Percentage
        df = df.with_columns(
            pl.when(pl.col("original_price").is_not_null())
            .then(((pl.col("original_price") - pl.col("promotion_price")) / pl.col("original_price") * 100).round(1))
            .otherwise(None)
            .alias("discount_pct")
        ).sort("name")

    return df

In [26]:
# @title RUN create dataframe
# --- RUN IT ---
# Notice we DO NOT add "&page={}" to the URL anymore. We let the clicker do the work!
url_neo_fineline = "https://www.makro.pro/en/c/collections/Shop%20by%20Brand:%20FINELINE/22199?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJtYXJrZXRpbmdDYXJvdXNlbCUyMiUyQyUyMmJhbm5lck5hbWUlMjIlM0ElMjJTaG9wJTIwYnklMjBCcmFuZCUzQSUyMEZJTkVMSU5FJTIyJTdE"

url_laundry = "https://www.makro.pro/en/c/household-supplies/laundry-supplies"

url_fresh_soft = "https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20Fresh%20and%20Soft/17985?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTJDJTIyY2Fyb3VzZWxUaXRsZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRnJlc2glMjBhbmQlMjBTb2Z0JTIyJTdE"

url_dishwash = "https://www.makro.pro/en/c/collections/Home%20Care%20%7C%20Dishwash%20Care/17988?info=JTdCJTIyc291cmNlRXZlbnQlMjIlM0ElMjJmbGV4aXBhZ2VDYXJvdXNlbCUyMiUyQyUyMmNhcm91c2VsTmFtZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRGlzaHdhc2glMjBDYXJlJTIyJTJDJTIyY2Fyb3VzZWxUaXRsZSUyMiUzQSUyMkhvbWUlMjBDYXJlJTIwJTdDJTIwRGlzaHdhc2glMjBDYXJlJTIyJTdE"



# ---
df_neo_fineline = await scrape_makro_spa_clicker(start_url=url_neo_fineline)

print("\n" + "=" * 60)
print(f"Makro Scraping Complete - {len(df_neo_fineline)} unique products")
print("=" * 60)
print(df_neo_fineline)
# ---
df_makro_detergent = await scrape_makro_spa_clicker(start_url=url_laundry)

print("\n" + "=" * 60)
print(f"Makro Scraping Complete - {len(df_makro_detergent)} unique products")
print("=" * 60)
print(df_makro_detergent)
# ---
df_makro_freshsoft = await scrape_makro_spa_clicker(start_url=url_fresh_soft)
print("\n" + "=" * 60)
print(f"Makro Scraping Complete - {len(df_makro_freshsoft)} unique products")
print("=" * 60)
print(df_makro_freshsoft)
# ---
df_makro_dishwash = await scrape_makro_spa_clicker(start_url=url_dishwash)
print("\n" + "=" * 60)
print(f"Makro Scraping Complete - {len(df_makro_dishwash)} unique products")
print("=" * 60)
print(df_makro_dishwash)

Walking through Makro's front door...
Scraping page 1...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 2...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 3...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 4...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 5...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 6...
  -> Extracted 20 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 7...
  -> Extracted 10 new products.
  -> Clicked 'Next'. Waiting for SPA to load...
Scraping page 8...
  -> Extracted 0 new products.
  -> Pagination stopped: Locator.click: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("text=Next").first
    - locator resolved to <button disabled tabindex="-1" type="button" data-test-id="cypress-compone

In [56]:
import asyncio
import polars as pl
from playwright.async_api import async_playwright

async def scrape_makro_product(url, browser_instance):
    """Modified to accept a browser instance to save resources"""
    context = await browser_instance.new_context(
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    page = await context.new_page()

    try:
        print(f"Scraping: {url}")
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(2000)

        async def get_text(selector):
            try:
                element = page.locator(selector).first
                return (await element.inner_text(timeout=5000)).strip()
            except:
                return None

        data = {
            # "url": url, # Added URL for tracking
            "name": await get_text('[data-test-id*="_product_title"]'),
            "promotion_price": await get_text('p.css-tn2bzw'),
            "original_price": await get_text('p.css-1krve4e'),
            "condition": await get_text('[data-test-id*="_lbl_buy_more"]')
        }
    except Exception as e:
        data = {"url": url, "error": str(e)}
    finally:
        await context.close()
    return data

async def main(urls):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        # This runs all scrapes concurrently
        tasks = [scrape_makro_product(url, browser) for url in urls]
        results = await asyncio.gather(*tasks)

        await browser.close()
        return results

# --- Execution ---
urls = [
    "https://www.makro.pro/en/p/262550-126255086281148",
    "https://www.makro.pro/en/p/z0icJlu-272067985587301",
]

# Run the async loop
raw_data = await main(urls)

# 1. Load into Polars
df_adhoc = pl.DataFrame(raw_data)

# 2. Polars-style cleaning (handling nulls and symbols)
df_adhoc = df_adhoc.with_columns([
    pl.col("original_price").str.replace("฿", "").str.strip_chars(),
    pl.col("promotion_price").str.strip_chars()
])

print("\n--- Final Polars DataFrame ---")
print(df_adhoc)

Scraping: https://www.makro.pro/en/p/z0icJlu-272067985587301
Scraping: https://www.makro.pro/en/p/262550-126255086281148

--- Final Polars DataFrame ---
shape: (2, 4)
┌─────────────────────────────────┬─────────────────┬────────────────┬─────────────┐
│ name                            ┆ promotion_price ┆ original_price ┆ condition   │
│ ---                             ┆ ---             ┆ ---            ┆ ---         │
│ str                             ┆ str             ┆ str            ┆ str         │
╞═════════════════════════════════╪═════════════════╪════════════════╪═════════════╡
│ PRO Regular Powder Detergent B… ┆ 139             ┆ 154            ┆ null        │
│ Fineline Liquid Detergent Plus… ┆ 116             ┆ 179            ┆ 2 - 2 units │
└─────────────────────────────────┴─────────────────┴────────────────┴─────────────┘


In [27]:
df_makro = pl.concat([df_neo_fineline, df_makro_detergent, df_makro_freshsoft, df_makro_dishwash, df_makro_PRO, df_adhoc])
## Transform Makro
if "discount_pct" in df_makro.columns:
    df_makro = df_makro.drop("discount_pct")

In [28]:
def re_evaluate_price(df: pl.DataFrame) -> pl.DataFrame:
    """
    Standardizes pricing logic:
    1. If original_price is missing, move the promotion_price to it.
    2. If promotion_price matches the original_price, set it to Null.
    """
    return (
        df.with_columns(
            # Step 1: Fix missing original prices by 'swapping' from promotion_price
            pl.when(pl.col("original_price").is_null() & pl.col("promotion_price").is_not_null())
            .then(pl.col("promotion_price"))
            .otherwise(pl.col("original_price"))
            .alias("original_price")
        )
        .with_columns(
            # Step 2: Now that original_price is populated, nullify redundant promotions
            pl.when(pl.col("promotion_price") == pl.col("original_price"))
            .then(None)
            .otherwise(pl.col("promotion_price"))
            .alias("promotion_price")
        )
    )

# How to use it:
df_prep_makro = re_evaluate_price(df_makro)

In [29]:
# @title udf Transform
def parse_product_names(df: pl.DataFrame, shop_name: str) -> pl.DataFrame:
    """
    Pass any supermarket dataframe through this function to standardize the columns.
    """
    # 1. Setup the dynamic date
    today_date = date.today().strftime("%Y-%m-%d")

    # 2. Define the patterns (Centralized here so you only ever update them in one place!)
    quant_unit_pattern = r"(?i)(\d+)\s*(ML|G|KG|L)\b"
    pack_pattern = r"(?i)(PACK\s*\d*\s*FREE\s*\d+|PACK\s*\d*\s*\+\s*\d+|PACK\s*\d+|TWINPACK|\bX\s*\d+\b|P?\d+\s*\+\s*\d+|\(\d+\+\d+\)|\d+\s*FREE\s*\d+|\bPACK\b)"

    # 3. Apply the Polars transformation
    return df.with_columns(
        pl.lit(today_date).alias("Date"),

        # Extract Brand
        pl.col("name").str.split(" ").list.first().alias("Brand"),

        # Extract Quantity (Pro-Tip: strict=False prevents the pipeline from crashing if a weird string can't be cast to an Integer)
        pl.col("name").str.extract(quant_unit_pattern, 1).cast(pl.Int64, strict=False).alias("Volume"),

        # Extract Unit
        pl.col("name").str.extract(quant_unit_pattern, 2).str.to_uppercase().alias("Unit"),

        # Extract Pack size
        pl.col("name").str.extract(pack_pattern, 1).str.to_uppercase().alias("Pack"),

        # Add the dynamic Shop identifier
        pl.lit(shop_name).alias("Retailer")
    )

In [30]:
df_trans_makro = parse_product_names(df_prep_makro, "Makro")

In [31]:
df_trans_makro

name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
str,f64,f64,str,str,str,i64,str,str,str
"""FINELINE CONCENTRATED FABRIC S…",113.0,169.0,"""2+ units""","""2026-04-10""","""FINELINE""",1150,"""ML""",null,"""Makro"""
"""FINELINE CONCENTRATED FABRIC S…",113.0,169.0,"""2+ units""","""2026-04-10""","""FINELINE""",1150,"""ML""",null,"""Makro"""
"""FINELINE CONCENTRATED FABRIC S…",53.0,69.0,"""2 - 2 units""","""2026-04-10""","""FINELINE""",450,"""ML""",null,"""Makro"""
"""FINELINE CONCENTRATED FABRIC S…",107.0,169.0,"""2 - 2 units""","""2026-04-10""","""FINELINE""",1150,"""ML""",null,"""Makro"""
"""FINELINE CONCENTRATED FABRIC S…",107.0,169.0,"""2 - 2 units""","""2026-04-10""","""FINELINE""",1150,"""ML""",null,"""Makro"""
…,…,…,…,…,…,…,…,…,…
"""SUNLIGHT Lemongreenlime 150 ml…",null,550.0,null,"""2026-04-10""","""SUNLIGHT""",150,"""ML""","""X 48""","""Makro"""
"""SUNLIGHT Plus Antibac 145 ml x…",null,59.0,null,"""2026-04-10""","""SUNLIGHT""",145,"""ML""","""X 6""","""Makro"""
"""SUNLIGHT Plus Dishwashing Liqu…",null,99.0,null,"""2026-04-10""","""SUNLIGHT""",7,"""L""",null,"""Makro"""


In [33]:
df_trans_makro.write_excel(f"makro_{today_date}.xlsx")

In [37]:
list_to_search = [
# # --- Lotus's
# "FINELINE LIQUID PLUS GOLD 550 ML.",
# "FINELINE LIQUID PLUS GOLD 1250 ML.",
# "HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 600 ML.",
# "HYGIENE LAUNDRY EXPERT WASH MILKY TOUCH 1400 ML.",
# "PAO WIN WASH LIQUID REFILL 620 ML",
# "PAO WIN WASH LIQUID 1300 ML",
# "PAO SUPER WHITE POWDER DETERGENT1800G.P2",
# "PAO SUPER WHITE POWDER DETERGENT 2400 G",
# "ATTACK EASY POWDER HAPPY SWEET 1800G. TWIN PACK",
# "ATTACK EASY HAPPY SWEET DETERGENT 2500G.",
# "PRO POWER DETERGENT BLUE PLUS 1700 G. PACK 1+1",
# "PRO POWDER DETERGENT BLUE PLUS 2400 G.",
# "HYGIENE FABRIC SOFTENER EXPERT CARE MILKY TOUCH 480 ML.",
# "HYGIENE FABRIC SOFTENER EXPERT CARE MILKY WHITE 480 ML 2+1",
# "HYGIENE FABRIC SOFT EXPERTCARE MILKY WHITE 1000 ML",
# "HYGIENE FABRIC SOFTENER EXPERTCARE MILKY TOUCH 1000 ML 1+1",
# "LIPON F DISHWASHING HYGIENE 500 ML PACK 3",
# "LIPON F DISH WASHER XTRA HYGENIC 750 ML. PACK 2",
# "LIPON-F DISHWASH BERGAMOT GALLON 3200 ML",
# # --- Big C ---
# "FINELINE Liquid Laundry Detergent Sunny Gold Scent 550 ml.",
# "FINELINE Plus Liquid Laundry Detergent Sunny Gold Scent 1250 ml.",
# "HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 600 ml.",
# "HYGIENE Expert Wash Concentrate Liquid Detergent Milky Touch 1400 ml.",
# "PAO Win Wash Liquid Laundry Detergent 620 ml.",
# "PAO Win Wash Liquid Laundry Detergent 1300 ml.",
# "PAO Super White Laundry Detergent 1800 g.",
# "PAO Super White Powder Laundry Detergent 2400 g.",
# "ATTACK Easy Conventional Detergent Happy Sweet Pink 1.7 kg x 1+1",
# "ATTACK EASY DETERGENT HAPPY SWEET 2500 G",
# "PRO Blue Plus Powder Laundry Detergent 2400 g.",
# "HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml.",
# "HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 480 ml. Pack 2+1",
# "HYGIENE Expert Care Concentrated Fabric Softener Milky Touch Scent 1000 ml. Pack 2",
# "LIPON F Dishwashing Liquid Hygienic Formula 500 ml. Pack 3",
# "LIPON F Sanitary Formula Dish Washing Liquid Refill 750 ml. Pack of 2",
# "LIPON F Dishwashing Liquid Hygienic Formula 3200 ml.",
# --- Makro ---
"Fineline Liquid Detergent Plus Sunny Gold 550 ML. Gold",
"Fineline Liquid Detergent Plus Sunny Gold 1250 ML.",
"HYGIENE Expert Wash Liauid Detergent Milky Touch 1.4 ml",
"PAO Win Wash Concentrated Liquid Detergent Orange 620 ml",
"PAO Win Wash Concentrated Liquid Detergent Orange 1.3 l",
"PAO Super White Laundry Detergent 1.8 kg x 1+1",
"PAO Super White Standard Formula Powder Detergent 2.4 kg",
"ATTACK Easy Regular Detergent Happy Sweet Pink 2.3/2.5 kg",
"PRO Regular Powder Detergent Blue Plus Red 1.7 kg x 1+1",
"PRO Regular Powder Detergent Blue Plus Red 2.4 kg",
"HYGIENE Fabric Softener Expert Care Milky Touch White 480 ml",
"HYGIENE EXPERT CARE CONCENTRATE FABRIC SOFTENER MILKY TOUCH 480 ML X 2+1 BAGS",
"HYGIENE Expert Care Concentrated Fabric Softener Milky Touch 1 l",
"HYGIENE Expert Care Concentrate Softener Duo Milky Touch White 1 l x 1+1",
"LIPON F Dishwash 500 ml x 3",
"LIPON F DISHWASHING LIQUID 3.2 L."
]

In [38]:
search_df = pl.DataFrame({"product_name": list_to_search})

makro_names_set = set(df_trans_makro["name"].to_list())

search_results_df = search_df.with_columns(
    pl.col("product_name").is_in(makro_names_set).alias("Found")
)

print("Search Results:")
print(search_results_df)


Search Results:
shape: (16, 2)
┌─────────────────────────────────┬───────┐
│ product_name                    ┆ Found │
│ ---                             ┆ ---   │
│ str                             ┆ bool  │
╞═════════════════════════════════╪═══════╡
│ Fineline Liquid Detergent Plus… ┆ true  │
│ Fineline Liquid Detergent Plus… ┆ false │
│ HYGIENE Expert Wash Liauid Det… ┆ true  │
│ PAO Win Wash Concentrated Liqu… ┆ true  │
│ PAO Win Wash Concentrated Liqu… ┆ true  │
│ …                               ┆ …     │
│ HYGIENE EXPERT CARE CONCENTRAT… ┆ true  │
│ HYGIENE Expert Care Concentrat… ┆ true  │
│ HYGIENE Expert Care Concentrat… ┆ true  │
│ LIPON F Dishwash 500 ml x 3     ┆ true  │
│ LIPON F DISHWASHING LIQUID 3.2… ┆ true  │
└─────────────────────────────────┴───────┘


In [39]:
search_results_df.to_pandas()

,product_name,Found
0,Fineline Liquid Detergent Plus Sunny Gold 550 ...,True
1,Fineline Liquid Detergent Plus Sunny Gold 1250...,False
2,HYGIENE Expert Wash Liauid Detergent Milky Tou...,True
3,PAO Win Wash Concentrated Liquid Detergent Ora...,True
4,PAO Win Wash Concentrated Liquid Detergent Ora...,True
5,PAO Super White Laundry Detergent 1.8 kg x 1+1,True
6,PAO Super White Standard Formula Powder Deterg...,True
7,ATTACK Easy Regular Detergent Happy Sweet Pink...,True
8,\tPRO Regular Powder Detergent Blue Plus Red 1...,False
9,PRO Regular Powder Detergent Blue Plus Red 2.4 kg,True


In [43]:
df_pro_brand = df_trans_makro.filter(pl.col("Brand").str.to_lowercase().is_in(["pro"]))
print("Products with Brand 'PRO':")
print(df_pro_brand.to_pandas())

Products with Brand 'PRO':
                                                 name  promotion_price  \
0    PRO Fabric Softener Garden Sweet Pink 450 ml x 3             25.0   
1    PRO Matic Standard Formula Powder Detergent 8 kg              NaN   
2   PRO REGULAR SOFTENER FOREST AROMA VIOLET 500 M...             25.0   
3        PRO REGULAR SOFTENER NATURAL BLUE 450 ML X 3             25.0   
4          PRO Regular Detergent Blue Plus Red 4.5 kg            189.0   
5      PRO Regular Detergent Sweet Floral Pink 2.4 kg             99.0   
6   PRO Regular Powder Detergent Blue Plus Red 2.4 kg             99.0   
7   PRO Regular Powder Detergent White Return Gree...             99.0   
8    PRO Fabric Softener Garden Sweet Pink 450 ml x 3             25.0   
9   PRO REGULAR SOFTENER FOREST AROMA VIOLET 500 M...             25.0   
10       PRO REGULAR SOFTENER NATURAL BLUE 450 ML X 3             25.0   
11                            PRO Dishwash 400 ml x 3             27.0   
12         

In [44]:
df_pro_brand.to_pandas()

,name,promotion_price,original_price,condition,Date,Brand,Volume,Unit,Pack,Retailer
0,PRO Fabric Softener Garden Sweet Pink 450 ml x 3,25.0,27.0,2+ units,2026-04-10,PRO,450,ML,X 3,Makro
1,PRO Matic Standard Formula Powder Detergent 8 kg,NaN,350.0,None,2026-04-10,PRO,8,KG,None,Makro
2,PRO REGULAR SOFTENER FOREST AROMA VIOLET 500 M...,25.0,27.0,2+ units,2026-04-10,PRO,500,ML,X 3,Makro
3,PRO REGULAR SOFTENER NATURAL BLUE 450 ML X 3,25.0,27.0,2+ units,2026-04-10,PRO,450,ML,X 3,Makro
4,PRO Regular Detergent Blue Plus Red 4.5 kg,189.0,212.0,None,2026-04-10,PRO,5,KG,None,Makro
5,PRO Regular Detergent Sweet Floral Pink 2.4 kg,99.0,143.0,None,2026-04-10,PRO,4,KG,None,Makro
6,PRO Regular Powder Detergent Blue Plus Red 2.4 kg,99.0,143.0,None,2026-04-10,PRO,4,KG,None,Makro
7,PRO Regular Powder Detergent White Return Gree...,99.0,143.0,None,2026-04-10,PRO,4,KG,None,Makro
8,PRO Fabric Softener Garden Sweet Pink 450 ml x 3,25.0,27.0,2+ units,2026-04-10,PRO,450,ML,X 3,Makro
9,PRO REGULAR SOFTENER FOREST AROMA VIOLET 500 M...,25.0,27.0,2+ units,2026-04-10,PRO,500,ML,X 3,Makro
